# 🎙️ tech-history 음성 생산 v3 — 숫자 발음 수정판

**이 노트북이 맞는지 확인:** 아래 두 번째 셀을 실행하면 `[v3] 숫자 한글 변환 켜짐` 이 출력됩니다.
(안 나오면 옛 사본입니다 — 이 파일을 깃허브에서 다시 여세요.)

**사용법 (순서 중요!)**
1. 런타임 → 런타임 유형 변경 → **T4 GPU**
2. **첫 코드 셀만** 실행 (설치, 2~3분)
3. 런타임 → **세션 다시 시작** (필수)
4. 런타임 → **모두 실행**
5. 끝나면 **`voice_01_fix.zip`** 이 자동 다운로드됨 (숫자 있는 3조각만 들어 있음)

In [ ]:
EPISODE = "01"  # 생산할 편 번호
!pip install -q chatterbox-tts requests
!pip uninstall -y -q torchvision  # 미사용 부품 — 구버전 torch와 충돌하므로 제거
print("설치 완료 — 이제 메뉴에서 [런타임 → 세션 다시 시작] 후 [모두 실행] 하세요")

In [ ]:
print("[v3] 숫자 한글 변환 켜짐")  # 수정판 확인 표식
import json, os, re, shutil, requests, torch, torchaudio
from chatterbox.mtl_tts import ChatterboxMultilingualTTS

url = f"https://raw.githubusercontent.com/nous-zero/tech-history/main/video/scripts/{EPISODE}.json"
script = requests.get(url).json()
print("대본:", script["title"], "/ 문단", len(script["segments"]))

device = "cuda" if torch.cuda.is_available() else "cpu"
print("장치:", device)
model = ChatterboxMultilingualTTS.from_pretrained(device=device)

In [ ]:
ONLY = [1]  # 이 번호 세그먼트만 재생산 — 현재: seg001 음성 뭉개짐 재생산 (빈 목록 [] 이면 전부)

# --- 숫자 → 한글 발음 변환 (TTS 입력 전용 — 화면 자막은 원문 숫자 유지) ---
_SINO = "영일이삼사오육칠팔구"

def _sino(n):
    n = int(n)
    if n == 0:
        return "영"
    out = ""
    for val, name in ((10000, "만"), (1000, "천"), (100, "백"), (10, "십"), (1, "")):
        d, n = n // val, n % val
        if d:
            out += ("" if d == 1 and name else _SINO[d]) + name
    return out

_MONTH = {6: "유", 10: "시"}  # 6월=유월, 10월=시월

def normalize_numbers(t):
    t = re.sub(r"(\d+)월", lambda m: (_MONTH.get(int(m.group(1))) or _sino(m.group(1))) + "월", t)
    return re.sub(r"\d+", lambda m: _sino(m.group(0)), t)

REF = "ref.wav" if os.path.exists("ref.wav") else None  # 육성 복제용(선택)
out_dir = f"voice_{EPISODE}_fix"
shutil.rmtree(out_dir, ignore_errors=True)  # 이전 실행 잔존 파일 제거 — 산출물은 이번 생산분만
os.makedirs(out_dir)
for seg in script["segments"]:
    if ONLY and seg["id"] not in ONLY:
        continue
    text = normalize_numbers(seg["text"])
    print(f"seg{seg['id']:03d} 읽을 문장: {text}")
    kwargs = {"language_id": "ko"}
    if REF:
        kwargs["audio_prompt_path"] = REF
    limit = 0.25 * len(text) + 5  # 비정상 길이(환각 반복) 감시선
    for attempt in range(3):
        wav = model.generate(text, **kwargs)
        sec = wav.shape[-1] / model.sr
        if sec <= limit:
            break
        print(f"  {sec:.1f}초 — 비정상(기준 {limit:.0f}초), 재시도 {attempt + 1}/3")
    torchaudio.save(os.path.join(out_dir, f"seg{seg['id']:03d}.wav"), wav, model.sr)
    print(f"seg{seg['id']:03d} 완료 ({sec:.1f}초)")
print("합성 완료:", "전체" if not ONLY else f"세그먼트 {ONLY}")

In [ ]:
import shutil
zip_path = shutil.make_archive(f"voice_{EPISODE}_fix", "zip", f"voice_{EPISODE}_fix")
from google.colab import files
files.download(zip_path)